# 03: TF-IDF Feature Extraction & Classical Machine Learning Models
**Owner:** Jeel (Person 2 - `jeel_p2`)
**Tasks:** P2.1 - P2.13

### Objective:
Extract TF-IDF features (unigram vs unigram+bigram), train Logistic Regression, Linear SVM, and Naive Bayes baselines.
Evaluate on validation set using Macro-F1 and per-class reports. Diagnose classification errors.

### Pipeline Overview:
1. Load & preprocess data
2. Stratified train/val/test split (70/15/15)
3. TF-IDF feature extraction comparison
4. Train 3 classical ML models
5. Cross-validation tuning
6. Full metrics report & comparison table
7. Confusion matrix visualization
8. Feature interpretation (top TF-IDF terms per class)
9. Error analysis & failure diagnosis
10. Save model pipeline

In [ ]:
# Setup
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import time
import json

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.pipeline import Pipeline

from src.data_loader import load_resume_csv, stratified_split
from src.preprocessing import clean_resume_text

print('All imports successful!')

## P2.1 - Data Loading, Cleaning & Stratified Split

In [ ]:
# Load dataset
df = load_resume_csv('../data/raw/csv/Resume.csv')
print(f'Loaded {len(df)} records across {df["Category"].nunique()} categories')
print(f'Columns: {list(df.columns)}')
print()
print(df['Category'].value_counts())

In [ ]:
# Clean text
df['cleaned_text'] = df['Resume_str'].apply(clean_resume_text)
df = df[df['cleaned_text'].str.strip().astype(bool)].reset_index(drop=True)
print(f'Records after cleaning: {len(df)}')

# Stratified split (70/15/15)
train_df, val_df, test_df = stratified_split(
    df, text_col='cleaned_text', label_col='Category',
    train_size=0.70, val_size=0.15, test_size=0.15, random_state=42
)
print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')

# Save splits
os.makedirs('../data/processed', exist_ok=True)
train_df.to_csv('../data/processed/train.csv', index=False)
val_df.to_csv('../data/processed/val.csv', index=False)
test_df.to_csv('../data/processed/test.csv', index=False)
print('Splits saved to data/processed/')

## P2.2 - TF-IDF Baseline: (1,1) vs (1,2) Comparison

In [ ]:
configs = [
    {'name': 'Unigram (1,1)',           'ngram_range': (1, 1), 'max_features': 10000, 'min_df': 2, 'max_df': 0.95},
    {'name': 'Unigram+Bigram (1,2)',    'ngram_range': (1, 2), 'max_features': 15000, 'min_df': 2, 'max_df': 0.95},
    {'name': 'Unigram+Bigram (1,2) 20k','ngram_range': (1, 2), 'max_features': 20000, 'min_df': 3, 'max_df': 0.90},
]

tfidf_results = {}
best_f1, best_config_name, best_vectorizer = -1, None, None

for cfg in configs:
    vec = TfidfVectorizer(
        ngram_range=cfg['ngram_range'], max_features=cfg['max_features'],
        min_df=cfg['min_df'], max_df=cfg['max_df'],
        stop_words='english', sublinear_tf=True
    )
    X_tr = vec.fit_transform(train_df['cleaned_text'])
    X_va = vec.transform(val_df['cleaned_text'])
    
    lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    lr.fit(X_tr, train_df['Category'])
    preds = lr.predict(X_va)
    mf1 = f1_score(val_df['Category'], preds, average='macro')
    acc = accuracy_score(val_df['Category'], preds)
    
    print(f"{cfg['name']:35s} | Features: {X_tr.shape[1]:5d} | Acc: {acc:.4f} | Macro-F1: {mf1:.4f}")
    tfidf_results[cfg['name']] = {'accuracy': acc, 'macro_f1': mf1, 'n_features': X_tr.shape[1]}
    
    if mf1 > best_f1:
        best_f1, best_config_name = mf1, cfg['name']
        best_vectorizer, best_X_train, best_X_val = vec, X_tr, X_va

print(f'\nBest TF-IDF config: {best_config_name} (Macro-F1={best_f1:.4f})')

## P2.3-P2.5 - Train Classical ML Models

In [ ]:
y_train = train_df['Category']
y_val = val_df['Category']
labels = sorted(y_train.unique().tolist())

models = {
    'Logistic_Regression': LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced', random_state=42, solver='lbfgs', multi_class='multinomial'),
    'Linear_SVM': SGDClassifier(loss='hinge', penalty='l2', alpha=1e-4, max_iter=2000, class_weight='balanced', random_state=42),
    'Multinomial_NB': MultinomialNB(alpha=0.1),
}

results = {}
for name, clf in models.items():
    t0 = time.time()
    clf.fit(best_X_train, y_train)
    train_time = time.time() - t0
    
    t0 = time.time()
    preds = clf.predict(best_X_val)
    pred_time = time.time() - t0
    
    acc = accuracy_score(y_val, preds)
    macro_f1 = f1_score(y_val, preds, average='macro', zero_division=0)
    weighted_f1 = f1_score(y_val, preds, average='weighted', zero_division=0)
    macro_p = precision_score(y_val, preds, average='macro', zero_division=0)
    macro_r = recall_score(y_val, preds, average='macro', zero_division=0)
    report = classification_report(y_val, preds, output_dict=True, zero_division=0)
    
    results[name] = {
        'model': clf, 'predictions': preds,
        'accuracy': acc, 'macro_precision': macro_p, 'macro_recall': macro_r,
        'macro_f1': macro_f1, 'weighted_f1': weighted_f1, 'report': report,
        'train_time_s': train_time, 'pred_time_ms': pred_time * 1000,
    }
    print(f'{name:25s} | Acc: {acc:.4f} | Macro-F1: {macro_f1:.4f} | Weighted-F1: {weighted_f1:.4f} | Train: {train_time:.2f}s')

best_model_name = max(results, key=lambda k: results[k]['macro_f1'])
print(f'\nBest Model: {best_model_name} (Macro-F1={results[best_model_name]["macro_f1"]:.4f})')

## P2.6 - Model Tuning via Cross-Validation

In [ ]:
X_train_full = best_vectorizer.fit_transform(train_df['cleaned_text'])
y_train_full = train_df['Category']

print('--- Logistic Regression C Tuning ---')
for c_val in [0.1, 0.5, 1.0, 5.0, 10.0]:
    lr = LogisticRegression(max_iter=2000, C=c_val, class_weight='balanced', random_state=42, solver='lbfgs', multi_class='multinomial')
    scores = cross_val_score(lr, X_train_full, y_train_full, cv=5, scoring='f1_macro', n_jobs=-1)
    print(f'  C={c_val:>5.1f} | 5-Fold Macro-F1: {scores.mean():.4f} +/- {scores.std():.4f}')

print('\n--- Linear SVM Alpha Tuning ---')
for alpha in [1e-5, 1e-4, 1e-3, 1e-2]:
    svm = SGDClassifier(loss='hinge', penalty='l2', alpha=alpha, max_iter=2000, class_weight='balanced', random_state=42)
    scores = cross_val_score(svm, X_train_full, y_train_full, cv=5, scoring='f1_macro', n_jobs=-1)
    print(f'  alpha={alpha:.0e} | 5-Fold Macro-F1: {scores.mean():.4f} +/- {scores.std():.4f}')

## P2.7 - Full Metrics Report

In [ ]:
for name, r in results.items():
    print(f'\n{"="*60}')
    print(f'{name}')
    print(f'{"="*60}')
    print(classification_report(y_val, r['predictions'], labels=labels, zero_division=0))

## P2.8 - Model Comparison Table

In [ ]:
rows = []
for name, r in results.items():
    rows.append({
        'Model': name, 'Accuracy': f"{r['accuracy']:.4f}",
        'Macro-P': f"{r['macro_precision']:.4f}", 'Macro-R': f"{r['macro_recall']:.4f}",
        'Macro-F1': f"{r['macro_f1']:.4f}", 'Weighted-F1': f"{r['weighted_f1']:.4f}",
        'Train(s)': f"{r['train_time_s']:.2f}", 'Pred(ms)': f"{r['pred_time_ms']:.1f}",
    })
comparison_df = pd.DataFrame(rows).sort_values('Macro-F1', ascending=False)
comparison_df

## P2.7 (cont.) - Confusion Matrix Heatmaps

In [ ]:
os.makedirs('../reports/figures', exist_ok=True)

for name, r in results.items():
    cm = confusion_matrix(y_val, r['predictions'], labels=labels)
    cm_norm = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-9)
    
    plt.figure(figsize=(16, 14))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=labels, yticklabels=labels, linewidths=0.5)
    plt.title(f'Confusion Matrix - {name}', fontsize=14, fontweight='bold')
    plt.xlabel('Predicted Category', fontsize=12)
    plt.ylabel('Actual Category', fontsize=12)
    plt.xticks(rotation=90, fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
    plt.tight_layout()
    plt.savefig(f'../reports/figures/confusion_matrix_{name}.png', dpi=200)
    plt.show()

## P2.9 - Top TF-IDF Terms Per Class

In [ ]:
best_model = results[best_model_name]['model']
feature_names = np.array(best_vectorizer.get_feature_names_out())

if hasattr(best_model, 'coef_'):
    for i, cls in enumerate(best_model.classes_):
        top_idx = np.argsort(best_model.coef_[i])[::-1][:10]
        terms = feature_names[top_idx]
        weights = best_model.coef_[i][top_idx]
        print(f'\n{cls}:')
        for t, w in zip(terms, weights):
            print(f'  {t:30s} | weight: {w:.4f}')

## P2.10-P2.11 - Error Analysis & Diagnosis

In [ ]:
best_preds = results[best_model_name]['predictions']
errors = []
for text, actual, pred in zip(val_df['cleaned_text'], y_val, best_preds):
    if actual != pred:
        errors.append({'Actual': actual, 'Predicted': pred, 'Snippet': str(text)[:200]})

error_df = pd.DataFrame(errors)
print(f'Total errors: {len(error_df)}/{len(y_val)} ({len(error_df)/len(y_val)*100:.1f}%)')
print()

if not error_df.empty:
    confusion_pairs = error_df.groupby(['Actual', 'Predicted']).size().reset_index(name='Count').sort_values('Count', ascending=False)
    print('Most Confused Category Pairs:')
    display(confusion_pairs)
    print()
    print('Error Details:')
    display(error_df)
else:
    print('No errors - perfect classification on validation set!')

## P2.12 - Save Model Pipeline

In [ ]:
os.makedirs('../models', exist_ok=True)

# Save vectorizer
joblib.dump(best_vectorizer, '../models/tfidf_vectorizer.joblib')

# Save best model
joblib.dump(best_model, f'../models/best_ml_model_{best_model_name}.joblib')

# Save full pipeline
pipeline = Pipeline([('tfidf', best_vectorizer), ('classifier', best_model)])
joblib.dump(pipeline, '../models/final_pipeline.joblib')

print(f'Saved vectorizer, model ({best_model_name}), and full pipeline to models/')
print(f'Best Model Macro-F1: {results[best_model_name]["macro_f1"]:.4f}')